# <Recipe title>

<One paragraph: what this does, for whom, and which Bodhan models it uses.>

**Requires** `BODHAN_API_KEY` in `.env` (see `.env.example`). Get a key at [console.bodhan.ai](https://console.bodhan.ai).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os, json, requests
from dotenv import load_dotenv

load_dotenv()
BASE_URL = os.environ.get("BODHAN_BASE_URL", "https://api.bodhan.ai")
HEADERS = {"Authorization": f"Bearer {os.environ['BODHAN_API_KEY']}"}


def bodhan(method: str, path: str, **kw):
    """Small wrapper: raises a readable error on non-2xx responses."""
    resp = requests.request(method, f"{BASE_URL}{path}", headers={**HEADERS, **kw.pop("headers", {})}, timeout=120, **kw)
    if not resp.ok:
        try:
            err = resp.json()["error"]
            raise RuntimeError(f"{resp.status_code} {err.get('code')}: {err.get('message')} (request_id={err.get('request_id')})")
        except (ValueError, KeyError):
            resp.raise_for_status()
    return resp

## Step 1 — <describe the first step>

In [ ]:
# Example: translate a sentence
out = bodhan("POST", "/translate", json={"text": "Hello, how are you?", "target_language": "hi"}).json()
out["translation"]

## Step 2 — <describe the next step>

In [ ]:
# Example: speak it (instructions is a JSON *string*)
audio = bodhan("POST", "/v1/audio/speech", json={
    "model": "indic-speak", "input": out["translation"], "voice": "Kavya", "instructions": json.dumps({"lang": "hi"}),
}).content
open("outputs/hello.wav", "wb").write(audio)

## Wrap-up

<What the reader has now, and one or two ideas for extending it.>

Before you commit: `make clean-outputs` and `python scripts/validate_recipe.py examples/<this_folder>` from the repo root.